# Virtual data set creation - standard L2 satellite swath format concatenated along a new time dimension


Demonstrates virtualization of a standard L2 satellite swath dataset - with 1D dimensions `along_track`, `cross_track` which have the same sizes accross all files. The L2 dataset used in this notebook is SCATSAT1_ESDR_L2_WIND_STRESS_V1.1.

SCATSAT1_ESDR_L2_WIND_STRESS_V1.1 is an L2 data set with no time dimension, so this is an example of saving along a newly created time dimension with values set to the granule start times for each timestamp. Granule start times are extracted from the UMM-G's. It turns out (as will be demo'd) that an L2 VDS structured in this way is sufficient to achieve both temporal and spatial search. For spatial search, this is achieved by first using `earthaccess` with spatial search to get the granules needed, then retrieving the granule start times from the metadata and using those values to subset the VDS along the newly created time dimension.

#### **Notes and Takeaways**
Xarray is trying to be too smart. It is using the CF-conventions to take the latitude, longitude variables and turn them into coordinates. This isn't necessarily a bad thing, but can lead to only the lat, lon values from the first file being used for the entire dataset - e.g. you have lat(along_track, cross_track) rather than the desired lat(along_track, cross_track, time). This can be avoided by being explicit about how to combine the coordinates in the call to `xarray.concat()`. This also brings up another point which is that in v1.x of virtualizarr, you cannot pass `decode_cf = False`, which would prevent the lat, lon vars from being turned into coordinates in the first place.

In [ ]:
# Built-in packages
import os
import sys
import shutil

# Filesystem management 
import fsspec
import earthaccess

# Data handling
import numpy as np
import xarray as xr
from virtualizarr import open_virtual_dataset
import pandas as pd

# Parallel computing 
import multiprocessing
from dask import delayed
import dask.array as da
from dask.distributed import Client
from distributed.utils import silence_logging_cmgr
#import coiled

# Other
#import matplotlib.pyplot as plt

## User-specific Inputs

In [4]:
shortname = "SCATSAT1_ESDR_L2_WIND_STRESS_V1.1"

In [5]:
# This will be assigned to 'loadable_variables' and needs to be modified per the specific 
# coord names of the data set. 
# !!Important for L2 swath datasets!!
# Don't assign latitude, longitude variables into 'loadable_variables' unless there is a specific reason! This will take
# a lot of memory and likely isn't what is required anyway.
coord_vars = []

In [6]:
concat_coords = ["lat", "lon"]  # Get's passed to xr.concat() when creating the combined VDS.

## 1. Get Data File S3 endpoints in Earthdata Cloud

In [12]:
# Get Earthdata creds
earthaccess.login()

Enter your Earthdata Login username:  deanh808
Enter your Earthdata password:  ········


In [13]:
# Get AWS creds. Note that if you spend more than 1 hour in the notebook, you may have to re-run this line!!!
fs = earthaccess.get_s3_filesystem(daac="PODAAC")

In [34]:
# Locate CCMP file information / metadata:
granule_info = earthaccess.search_data(
    short_name=shortname,
    )

In [35]:
# Get S3 endpoints for all files:
data_s3links = [g.data_links(access="direct")[0] for g in granule_info]
print(len(data_s3links))
data_s3links[0:3]

13145


['s3://podaac-ops-cumulus-protected/SCATSAT1_ESDR_L2_WIND_STRESS_V1.1/2018/091/measures_esdr_scatsat_l2_wind_stress_08002_v1.1_s20180401-011908-e20180401-025826.nc',
 's3://podaac-ops-cumulus-protected/SCATSAT1_ESDR_L2_WIND_STRESS_V1.1/2018/091/measures_esdr_scatsat_l2_wind_stress_08003_v1.1_s20180401-025826-e20180401-043745.nc',
 's3://podaac-ops-cumulus-protected/SCATSAT1_ESDR_L2_WIND_STRESS_V1.1/2018/091/measures_esdr_scatsat_l2_wind_stress_08004_v1.1_s20180401-043745-e20180401-061703.nc']

## 2. Generate single-orbit reference files

One file per orbit, so one reference file per orbit.

In [36]:
reader_opts = {"storage_options": fs.storage_options} # S3 filesystem creds from previous section.

#### Using local dask cluster for scaling

In [17]:
print("CPU count =", multiprocessing.cpu_count())

CPU count = 32


In [18]:
# Start up cluster and print some information about it:
client = Client(n_workers=30, threads_per_worker=1)
print(client.cluster)
print("View any work being done on the cluster here", client.dashboard_link)

LocalCluster(60e080ff, 'tcp://127.0.0.1:45781', workers=30, threads=30, memory=121.90 GiB)
View any work being done on the cluster here https://cluster-ynjiz.dask.host/jupyter/proxy/8787/status


2025-12-09 00:17:12,090 - distributed.scheduler - WARNING - Detected different `run_spec` for key 'original-open_dataset-wind_stress_magnitude-3929cf9cca2b1c4802c25733ab6f3aac' between two consecutive calls to `update_graph`. This can cause failures and deadlocks down the line. Please ensure unique key names. If you are using a standard dask collections, consider releasing all the data before resubmitting another computation. More details and help can be found at https://github.com/dask/dask/issues/9888. 
Debugging information
---------------------
old task state: released
old run_spec: (<function execute_task at 0x7eb996dfec00>, (ImplicitToExplicitIndexingAdapter(array=CopyOnWriteArray(array=LazilyIndexedArray(array=_ElementwiseFunctionArray(LazilyIndexedArray(array=<xarray.backends.zarr.ZarrArrayWrapper object at 0x7eb96cad1e00>, key=BasicIndexer((slice(None, None, None), slice(None, None, None), slice(None, None, None)))), func=functools.partial(<function _apply_mask at 0x7eb99cabb6

In [21]:
%%time
# Create individual references:
open_vds_par = delayed(open_virtual_dataset)
tasks = [
    open_vds_par(p, indexes={}, reader_options=reader_opts, loadable_variables=coord_vars, decode_times=False) 
    for p in data_s3links
    ]
virtual_ds_list = list(da.compute(*tasks)) # The xr.combine_nested() function below needs a list rather than a tuple.

CPU times: user 3min 2s, sys: 36.6 s, total: 3min 38s
Wall time: 8min


## 3. Generate combined reference file

**Combine along a newly created time dimension:** The dimension will be "orbit start time", so get the start time from each file using the granule metadata. Convert those times into cf-compliant formats (e.g. "second since ...").

In [25]:
## The dimension will be "orbit start time", so get the start time from each file using the granule metadata:
## This block of code assumes that the datetime string stored in the UMM-G ends with "Z" for UTC timezone.

basetime_str = "1970-01-01T00:00:00" # times will be measured in seconds since this basetime UTC. 

orbit_starttime_list = []
for g in granule_info:
    datetime_str = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'][:-1]  # -1 to remove "Z" at end.
    datetime_obj = np.datetime64(datetime_str)
    basetime_obj = np.datetime64(basetime_str)
    timedelt = np.timedelta64(datetime_obj - basetime_obj, 's').astype(int)
    orbit_starttime_list.append(timedelt)

In [26]:
## Wrap the orbit start time data in an xarray.DataArray, assigning CF-aligned attributes:
orbit_starttime_da = xr.DataArray(
    data=orbit_starttime_list,
    name="orbit_segment_start_time",
    dims=["orbit_segment_start_time"],
    attrs=dict(
        units="seconds since " + basetime_str,
        calendar = "gregorian"
    )
)

In [27]:
%%time
# Create the combined reference
virtual_ds_combined = xr.concat(
    virtual_ds_list, orbit_starttime_da, 
    coords = concat_coords, compat='override', combine_attrs='drop_conflicts'
)

CPU times: user 1min 8s, sys: 1.23 s, total: 1min 10s
Wall time: 1min 8s


In [28]:
virtual_ds_combined

<xarray.Dataset> Size: 896GB
Dimensions:                        (orbit_segment_start_time: 13145,
                                    along_track: 3248, cross_track: 152)
Coordinates:
    lat                            (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    lon                            (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
  * orbit_segment_start_time       (orbit_segment_start_time) int64 105kB 152...
Dimensions without coordinates: along_track, cross_track
Data variables: (12/34)
    real_wind_u_error              (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    flags                          (orbit_segment_start_time, along_track, cross_track) int32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=int32, ...
    quality_indicator              (orbit_segment_start_time, along_track, cross_track) int16 13GB ManifestArray<shape=(13145, 3248, 152), dtype=int16, ...
    en_wind_speed                  (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    wind_stress_u                  (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    en_wind_direction_error        (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    ...                             ...
    time                           (orbit_segment_start_time, along_track) float64 342MB ManifestArray<shape=(13145, 3248), dtype=float64, chunk...
    en_wind_u_error                (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    en_wind_direction              (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    en_wind_u                      (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    rain_speed_bias                (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
    nudge_wind_speed               (orbit_segment_start_time, along_track, cross_track) float32 26GB ManifestArray<shape=(13145, 3248, 152), dtype=float32...
Attributes: (12/27)
    Conventions:               CF-1.8, ACDD-1.3, ISO-8601
    ShortName:                 SCATSAT1_ESDR_L2_WIND_STRESS_V1.1
    creator_name:              Svetla Hristova-Veleva, Bryan Stiles, Alexande...
    creator_type:              institution
    geospatial_lat_max:        89.99N
    geospatial_lat_min:        -89.99N
    ...                        ...
    standard_name_vocabulary:  CF Standard Name Table v78
    summary:                   This project is funded by NASA’s Making Earth ...
    time_coverage_duration:    P0DT1H41M0S
    time_coverage_resolution:  P0DT0H0M30S
    title:                     SCATSAT-1 Scatterometer Inter-Calibrated ESDR ...
    version_id:                1.1

In [29]:
# Save in JSON format:
fname_combined_json = shortname + "_virtual_s3_concat-time.json"
virtual_ds_combined.virtualize.to_kerchunk(fname_combined_json, format='json')